In [1]:
# torch and modules
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Main architecture
from Architecture import Architecture

# custom utils and modules
from modules.trainer import Trainer
from modules.tester import test
from modules.charts import gen_line_charts

In [2]:
image_size = 64

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor()
])

# make test sets
test_datasets = datasets.ImageFolder("./datasets/test", transform)
test_loader = DataLoader(
    dataset=test_datasets, 
    batch_size=32,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

model = Architecture()

In [3]:
def add_conv_layers(layers=1, skip_pool=0):
    # define in and out channels
    in_channels = 3
    out_channels = 8
    # size
    size = image_size
    # skip pool increase by one for calculation
    skip_pool = skip_pool+1
    # save a trainable parameters
    total_conv_params = 0
    # loop each layers to add on model
    for layer in range(layers):
        # Convolutional Layer
        model.add(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )
        # calculate trainable params
        total_conv_params += (((3*3*in_channels) + 1) * out_channels) + (2*out_channels)
        # Pooling Layer
        if  (layer + 1) % skip_pool:
            model.add(nn.MaxPool2d(2,2))
            size = size//2
        # update in and out channels
        if (layer<layers-1):
            in_channels = out_channels
            out_channels = out_channels*2
    # return total convolutional layer parameters
    return total_conv_params, out_channels, size

In [4]:
conv_params, out_channels, size = add_conv_layers(8, 1)

# print conv params and out_channels
print(f"Total Trainable Parameters: {conv_params}")
print(f"Final Features: {out_channels}")
print(f"Final Feature size: {size}x{size}")

Total Trainable Parameters: 6297408
Final Features: 1024
Final Feature size: 4x4


In [5]:
# calculate input
n_in = out_channels*size*size
print(f"Input vector size = {n_in}")

model.add(
    # Flatten
    nn.Flatten(),
    # Hidden Layer
    nn.Linear(n_in, 128),
    nn.ReLU(),
    # Dropout
    nn.Dropout(0.3),
    # Output Layer
    nn.Linear(128, 3),
    nn.ReLU()
)

Input vector size = 16384


In [6]:
total_trainable_parameters = ((n_in*128)+128) + ((128*3)+3)
print(f"Total trainable parameters = {total_trainable_parameters}")

Total trainable parameters = 2097667


In [7]:
criterion = nn.CrossEntropyLoss()

In [8]:
tested = test(model, "./experiments/e7_ext_e5_dropout/models/epoch_84.pt", test_loader, criterion, "cuda")
tested

{'test_accuracy': 0.8728070175438597,
 'test_precision': 0.8738455076166066,
 'test_recall': 0.8735589713131567,
 'test_f1_score': 0.8736956362617136,
 'test_loss': 0.360955564926068}